# BioJEPA v0.6 Hyperparameter Optimization

Two-phase Bayesian HPO for pretraining hyperparameters:
- **Phase 1**: Exploration on balanced 50% data subsample (15 trials)
- **Phase 2**: Refinement on full data, warm-started from Phase 1 top configs (25 trials)

See `bayes_opt.md` for design rationale.

## Device & Paths

In [1]:
import torch
import random
from pathlib import Path
from collections import Counter
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from functools import partial
import numpy as np
import gc
import sys
sys.path.insert(0, str(Path.cwd().parent))

from biojepa_v0_6 import BioJepaConfig
from training_v0_6 import create_model
from dataloader_v0_6 import PretrainLoader
from evals.evals import EvalContext
from bayesian_optimization.hpo_utils import (
    is_valid_config, compute_step_budget, derive_parameters,
    check_identity_shortcut_cheap, check_identity_shortcut_full,
    check_cell_type_collapse, check_vicreg_collapse, check_perturbation_signal,
    compute_objective, create_phase1_subsample,
    run_selected_evals, SubsampledPretrainLoader
)

In [2]:
def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(1337)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(1337)
random.seed(1337)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('/home/ubuntu/data/v0_6')
ref_root = Path('/home/ubuntu/data/reference_data')
hpo_output_dir = data_root / 'hpo'
hpo_output_dir.mkdir(exist_ok=True)

using cuda


In [3]:
hpo_config = {
    'study_name': 'biojepa_v0_6_hpo_v6',
    'seed': 1337,
    'batch_size': 16,
    'test_total_examples': 5000,
    'verbose': False,
    'n_startup_trials': 10,
    'multivariate': True,
    'constant_liar': True,
    'pruner_startup_trials': 10,
    'pruner_warmup_steps': 5,
    'phase1_fraction': 0.5,
    'phase1_balance': 'equal',
    'cheap_eval_interval': 10000,
    'full_eval_interval': 30000,
    'check_warmup_steps': 30000,
    'embed_dim_options': [64, 128, 192, 256, 384],
    'n_layer_options': [4, 6, 8],
    'heads_options': [2, 4, 8],
    'n_pre_layer_ratio_range': (0.25, 0.75),
    'mask_ratio_range': (0.4, 0.85),
    'gaussian_scale_range': (1.0, 4.0),
    'ema_momentum_range': (0.985, 0.9995),
    'vicreg_weight_range': (0.5, 2.0),
    'film_linear_multiple_range': (0.5, 2.0),
    'lr_range': (5e-5, 3e-3),
    'num_genes': 10000,
    'mlp_ratio': 4.0,
    'use_amp': USE_AMP,
    'use_fused': USE_FUSED,
    'cheap_evals': ['batch_invariance', 'perturbation_detection', 'latent_space_health', 'reconstruction'],
    'full_evals': ['batch_invariance', 'perturbation_detection', 'latent_space_health', 'reconstruction',
                   'gene_embedding_pathways', 'essential_gene_prediction', 'embedding_consistency', 'cell_type_probing'],
}

In [4]:
def create_study(cfg, phase=1):
    study_name = f'{cfg["study_name"]}_phase{phase}'
    sampler = TPESampler(
        n_startup_trials=cfg['n_startup_trials'],
        multivariate=cfg['multivariate'],
        constant_liar=cfg['constant_liar'],
        seed=cfg['seed']
    )
    pruner = MedianPruner(
        n_startup_trials=cfg['pruner_startup_trials'],
        n_warmup_steps=cfg['pruner_warmup_steps'],
        interval_steps=1
    )
    return optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=pruner,
        study_name=study_name,
        storage=f'sqlite:///{hpo_output_dir / study_name}.db',
        load_if_exists=True
    )

In [5]:
def create_hpo_model(params, cfg, device):
    '''Create BioJepa from HPO params with correct config field names.'''
    derived = derive_parameters(params)
    vicreg_w = params['vicreg_weight']

    model_cfg = BioJepaConfig(
        num_genes=cfg['num_genes'],
        n_layer=params['n_layer'],
        heads=params['heads'],
        embed_dim=params['embed_dim'],
        mlp_ratio=cfg['mlp_ratio'],
        n_pre_layer=derived['n_pre_layer'],
        mask_ratio=params['mask_ratio'],
        gaussian_scale=params['gaussian_scale'],
        film_linear_multiple=params['film_linear_multiple'],
        ema_momentum=params['ema_momentum'],
        sim_coeff=25.0 * vicreg_w,
        std_coeff=25.0 * vicreg_w,
        cov_coeff=1.0 * vicreg_w,
    )
    print(model_cfg)
    return create_model(model_cfg, device), model_cfg

In [6]:
def summarize_eval_results(eval_results, embed_dim):
    '''Extract key metrics from eval results for logging.'''
    summary = {}
    if 'perturbation_detection' in eval_results:
        summary['pert_auroc'] = eval_results['perturbation_detection'].get('metrics', {}).get('auroc')
    if 'latent_space_health' in eval_results:
        health = eval_results['latent_space_health']
        summary['eff_dim_90'] = health.get('effective_dimensionality', {}).get('90_percent')
        summary['dead_dims'] = health.get('variance', {}).get('n_dead_dims')
    if 'reconstruction' in eval_results:
        summary['recon_pearson'] = eval_results['reconstruction'].get('metrics', {}).get('pearson_r')
    if 'batch_invariance' in eval_results:
        summary['invariance_ratio'] = eval_results['batch_invariance'].get('invariance_ratio')
    if 'gene_embedding_pathways' in eval_results:
        summary['kegg_silhouette'] = eval_results['gene_embedding_pathways'].get('kegg', {}).get('silhouette_score')
    if 'cell_type_probing' in eval_results:
        summary['cell_type_acc'] = eval_results['cell_type_probing'].get('metrics', {}).get('accuracy')
    if 'essential_gene_prediction' in eval_results:
        summary['essential_auroc'] = eval_results['essential_gene_prediction'].get('classification', {}).get('auroc_test')
    if 'embedding_consistency' in eval_results:
        summary['consistency_ratio'] = eval_results['embedding_consistency'].get('metrics', {}).get('inter_intra_ratio')
    return {k: round(v, 4) if isinstance(v, float) else v for k, v in summary.items() if v is not None}


def objective(trial, cfg, phase=1):
    params = {
        'embed_dim': trial.suggest_categorical('embed_dim', cfg['embed_dim_options']),
        'n_layer': trial.suggest_categorical('n_layer', cfg['n_layer_options']),
        'n_pre_layer_ratio': trial.suggest_float('n_pre_layer_ratio', *cfg['n_pre_layer_ratio_range']),
        'heads': trial.suggest_categorical('heads', cfg['heads_options']),
        'mask_ratio': trial.suggest_float('mask_ratio', *cfg['mask_ratio_range']),
        'gaussian_scale': trial.suggest_float('gaussian_scale', *cfg['gaussian_scale_range']),
        'ema_momentum': trial.suggest_float('ema_momentum', *cfg['ema_momentum_range']),
        'vicreg_weight': trial.suggest_float('vicreg_weight', *cfg['vicreg_weight_range']),
        'film_linear_multiple': trial.suggest_float('film_linear_multiple', *cfg['film_linear_multiple_range']),
        'lr': trial.suggest_float('lr', *cfg['lr_range'], log=True),
    }

    if not is_valid_config(params):
        trial.set_user_attr('fail_reason', f'Invalid config: head_dim={params["embed_dim"] // params["heads"]}')
        raise optuna.TrialPruned('Invalid configuration')

    embed_dim = params['embed_dim']
    pretraining_dir = data_root / 'pretraining'
    model, train_loader, eval_ctx = None, None, None
    optimizer, scheduler = None, None

    try:
        model, model_cfg = create_hpo_model(params, cfg, device)

        if phase == 1:
            shard_paths = create_phase1_subsample(
                pretraining_dir, cfg['phase1_fraction'], cfg['seed'], cfg['phase1_balance']
            )
            train_loader = SubsampledPretrainLoader(cfg['batch_size'], shard_paths, device)
        else:
            train_loader = PretrainLoader(cfg['batch_size'], 'train', pretraining_dir, device)

        step_budget = compute_step_budget(embed_dim, cfg['batch_size'])
        check_warmup = cfg.get('check_warmup_steps', 30000)
        report_step = 0
        current_step = 0

        eval_config = {
            'num_genes': cfg['num_genes'],
            'embed_dim': embed_dim,
            'n_layer': params['n_layer'],
            'heads': params['heads'],
            'batch_size': cfg['batch_size'],
            'test_total_examples': cfg['test_total_examples'],
            'verbose': cfg['verbose'],
        }

        model.enable_pretraining_gradients()
        pretraining_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(pretraining_params, lr=params['lr'], weight_decay=0.05, fused=cfg['use_fused'])
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=params['lr'], total_steps=step_budget, pct_start=0.05)

        use_amp = cfg['use_amp'] and device.type == 'cuda'

        model.train()
        while current_step < step_budget:
            steps_to_train = min(cfg['cheap_eval_interval'], step_budget - current_step)
            for st in range(steps_to_train):
                batch = train_loader.next_batch()
                optimizer.zero_grad()
                with torch.autocast('cuda', dtype=torch.bfloat16, enabled=use_amp):
                    loss = model.forward_pretrain(batch.x, batch.total)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(pretraining_params, 1.0)
                optimizer.step()
                model.update_teacher()
                scheduler.step()
                if st % 250 == 0: print(f'step {st}/{steps_to_train} | loss:{loss.item():.4f}')
            current_step += steps_to_train

            is_full_eval = (current_step % cfg['full_eval_interval'] == 0)
            evals_to_run = cfg['full_evals'] if is_full_eval else cfg['cheap_evals']

            eval_ctx = EvalContext.from_trained_model(model, data_root, ref_root, eval_config)
            eval_results = run_selected_evals(eval_ctx, evals_to_run)

            trial.set_user_attr(f'eval_{current_step}', summarize_eval_results(eval_results, embed_dim))

            if current_step >= check_warmup:
                fail_checks = [
                    check_vicreg_collapse(eval_results, embed_dim),
                    check_perturbation_signal(eval_results),
                    check_identity_shortcut_cheap(eval_results),
                ]
                for should_fail, reason in fail_checks:
                    if should_fail:
                        trial.set_user_attr('fail_reason', reason)
                        raise optuna.TrialPruned(reason)

            if is_full_eval:
                if current_step >= check_warmup:
                    fail_checks = [
                        check_identity_shortcut_full(eval_results),
                        check_cell_type_collapse(eval_results),
                    ]
                    for should_fail, reason in fail_checks:
                        if should_fail:
                            trial.set_user_attr('fail_reason', reason)
                            raise optuna.TrialPruned(reason)

                score = compute_objective(eval_results, embed_dim)
                trial.report(float(score), report_step)
                report_step += 1
                if trial.should_prune():
                    raise optuna.TrialPruned()

            del eval_ctx, eval_results
            eval_ctx = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            model.enable_pretraining_gradients()
            model.train()

        model.eval()
        eval_ctx = EvalContext.from_trained_model(model, data_root, ref_root, eval_config)
        final_results = run_selected_evals(eval_ctx, cfg['full_evals'])
        trial.set_user_attr('eval_final', summarize_eval_results(final_results, embed_dim))
        return float(compute_objective(final_results, embed_dim))

    except optuna.TrialPruned:
        raise
    except Exception as e:
        import traceback
        trial.set_user_attr('fail_reason', f'{type(e).__name__}: {e}')
        trial.set_user_attr('traceback', traceback.format_exc())
        raise

    finally:
        del eval_ctx, model, train_loader, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
phase1_study = create_study(hpo_config, phase=1)
phase1_study.optimize(
    partial(objective, cfg=hpo_config, phase=1),
    n_trials=15,
    show_progress_bar=True
)

completed_trials = [t for t in phase1_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if not completed_trials:
    print('WARNING: No completed trials in Phase 1. Phase 2 will start without warm-start.')
    top_params = []
else:
    print(f'Phase 1 Best trial: {phase1_study.best_trial.number}')
    print(f'Phase 1 Best score: {phase1_study.best_value:.4f}')
    print(f'Phase 1 Best params: {phase1_study.best_params}')
    top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:5]
    top_params = [t.params for t in top_trials]
    print(f'Warm-starting Phase 2 with {len(top_params)} configs from Phase 1')

/tmp/ipykernel_36540/2449546974.py:3: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = TPESampler(
/tmp/ipykernel_36540/2449546974.py:3: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  sampler = TPESampler(
[I 2026-02-06 17:23:23,753] A new study created in RDB with name: biojepa_v0_6_hpo_v6_phase1


  0%|          | 0/15 [00:00<?, ?it/s]

BioJepaConfig(num_genes=10000, n_layer=8, heads=8, embed_dim=256, mlp_ratio=4.0, n_pre_layer=5, mask_ratio=0.45627606685101973, gaussian_scale=3.9506458154309234, film_linear_multiple=1.69117786368738, sim_coeff=42.108437825896374, std_coeff=42.108437825896374, cov_coeff=1.6843375130358549, pert_latent_dim=320, pert_mode_dim=64, ema_momentum=0.9914267605953544)
step 0/10000 | loss:127.2054


In [ ]:
phase1_study = create_study(hpo_config, phase=1)

for trial in phase1_study.trials:
    state = trial.state.name
    reason = trial.user_attrs.get('fail_reason', 'N/A')
    params = {k: trial.params.get(k, '?') for k in ['embed_dim', 'n_layer', 'heads']}
    print(f'Trial {trial.number}: {state} | D={params["embed_dim"]} L={params["n_layer"]} H={params["heads"]} | {reason}')

states = Counter(t.state.name for t in phase1_study.trials)
reasons = Counter(t.user_attrs.get('fail_reason', 'N/A') for t in phase1_study.trials if t.state.name != 'COMPLETE')
print(f'\nState counts: {dict(states)}')
print(f'Fail reasons: {dict(reasons)}')

In [ ]:
phase2_study = create_study(hpo_config, phase=2)

for params in top_params:
    phase2_study.enqueue_trial(params)

phase2_study.optimize(
    partial(objective, cfg=hpo_config, phase=2),
    n_trials=25,
    show_progress_bar=True
)

completed_trials = [t for t in phase2_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if completed_trials:
    print(f'Phase 2 Best trial: {phase2_study.best_trial.number}')
    print(f'Phase 2 Best score: {phase2_study.best_value:.4f}')
    print(f'Phase 2 Best params: {phase2_study.best_params}')
else:
    print('WARNING: No completed trials in Phase 2.')

In [ ]:
df = phase2_study.trials_dataframe()
if df.empty:
    'No trials recorded.'
else:
    df.to_csv(hpo_output_dir / f'{hpo_config["study_name"]}_results.csv', index=False)
    completed_df = df[df['value'].notna()]
    if completed_df.empty:
        df[['number', 'state', 'value', 'params_embed_dim', 'params_n_layer', 'params_heads']]
    else:
        completed_df.nlargest(5, 'value')[['number', 'value', 'params_embed_dim', 'params_n_layer', 'params_mask_ratio']]